# 0. Project Overview

This report analyzes non-anomalous corporate network traffic to establish baseline behavior for SIEM rule development. Using dataset 10 (data10.parquet) containing verified legitimate flows, we identified normal communication patterns, bandwidth usage, and typical destinations.

We developed detection rules targeting five attack vectors: internal botnet activity, HTTPS/DNS data exfiltration, command-and-control attacks, anomalous external destinations, and suspicious external access patterns. Rules were validated against test datasets containing potential malicious activity to identify compromised devices.

The analysis utilized Python/pandas for parquet log processing and GeoLite2 databases for geographic context of external connections.

In [ ]:
import os
import json
import ipaddress
import numpy as np
import pandas as pd
import dns.resolver
import geoip2.database
import dns.reversename
import matplotlib.pyplot as plt

dataset_id = 10

script_directory = os.getcwd()
datasets_directory = os.path.join(script_directory, "datasets")
databases_directory = os.path.join(script_directory, "databases")
output_directory = os.path.join(script_directory, "output")

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

normal_flows_file = os.path.join(datasets_directory, f"data{dataset_id}.parquet")
anomalous_flows_file = os.path.join(datasets_directory, f"test{dataset_id}.parquet")
server_flows_file = os.path.join(datasets_directory, f"servers{dataset_id}.parquet")

geoASN = geoip2.database.Reader(
    os.path.join(script_directory, "databases/geolite2-asn.mmdb")
)
geoCC = geoip2.database.Reader(
    os.path.join(script_directory, "databases/geolite2-country.mmdb")
)

normal_data = pd.read_parquet(normal_flows_file)
anomalous_data = pd.read_parquet(anomalous_flows_file)
servers_data = pd.read_parquet(server_flows_file)

# 1. Normal Traffic Analysis

Baseline analysis of anomaly-free traffic is essential for detecting security threats through behavioral deviations. This chapter examines normal traffic patterns including protocol/port usage, flow distributions, IP behaviors, and server/client identification.

## 1.1. Network Topology Analysis and Server-Client Identification

Network analysis identified 192.168.110.0/24 as the main network with 197 unique IP addresses. Since source IPs are exclusively local while destinations include both local and external addresses, local destinations represent internal servers. We identified 6 unique local destination IPs functioning as internal servers, with services determined by their protocol:port combinations:

In [ ]:
def analyze_network_infrastructure(normal_data):
    """
    Analyze network infrastructure to identify subnets and device counts.
    
    Parameters:
    -----------
    normal_data : pandas.DataFrame
        The baseline network data containing 'src_ip' column
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with network subnets and device counts
    """
    import pandas as pd
    import ipaddress

    unique_ips = normal_data['src_ip'].unique()
    networks = {}
    
    for ip in unique_ips:
        try:
            ip_obj = ipaddress.IPv4Address(ip)
            if ip_obj.is_private:
                network = ipaddress.IPv4Network(f"{ip}/24", strict=False)
                network_str = str(network)
                
                if network_str not in networks:
                    networks[network_str] = set()
                networks[network_str].add(ip)
        except:
            continue
    
    results = []
    for network, devices in networks.items():
        results.append({
            'Network Subnet': network,
            'Device Count': len(devices)
        })
    
    network_df = pd.DataFrame(results).sort_values('Device Count', ascending=False)
    
    return network_df

analyze_network_infrastructure(normal_data)

In [ ]:
def analyze_internal_services(normal_data):
    """
    Analyze internal services by identifying private destination IPs and their characteristics.
    
    Parameters:
    -----------
    normal_data : pandas.DataFrame
        The baseline network data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with Server IP, Flow Count, and Protocols columns
    """
    import pandas as pd
    import ipaddress
    
    n_private_dst_data = normal_data[
        normal_data["dst_ip"].apply(lambda x: ipaddress.ip_address(x).is_private)
    ]
    
    n_private_dst_data_with_combo = n_private_dst_data.copy()
    n_private_dst_data_with_combo["proto_port"] = (
        n_private_dst_data_with_combo["proto"]
        + ":"
        + n_private_dst_data_with_combo["port"].astype(str)
    )
    
    private_dst_summary = (
        n_private_dst_data_with_combo.groupby("dst_ip")
        .agg({
            "proto_port": lambda x: sorted(set(x)),
            "dst_ip": "size"
        })
        .rename(columns={
            "proto_port": "Protocols", 
            "dst_ip": "Flow Count"
        })
        .sort_values("Flow Count", ascending=False)
    )
    
    services_df = private_dst_summary.reset_index()
    services_df = services_df.rename(columns={"dst_ip": "Server IP"})
    
    services_df["Protocols"] = services_df["Protocols"].apply(lambda x: ", ".join(x))
     
    return services_df

analyze_internal_services(normal_data)

## 1.2. Normal Traffic Characteristics

In [ ]:
def analyze_protocol_port_distribution(normal_data):
    """
    Analyze protocol:port combination distribution and additional network metrics.
    
    Parameters:
    -----------
    normal_data : pandas.DataFrame
        The baseline network data containing 'proto', 'port', 'src_ip', 'up_bytes', 'down_bytes' columns
    
    Returns:
    --------
    dict
        Dictionary containing all calculated metrics and formatted strings
    """
    import pandas as pd
    import numpy as np
    
    results = {}
    
    # 1. Protocol Distribution and Port Usage
    proto_dist = normal_data['proto'].value_counts()
    total_flows = len(normal_data)
    
    protocol_breakdown = []
    for proto, count in proto_dist.items():
        percentage = (count / total_flows) * 100
        protocol_breakdown.append(f"{percentage:.1f}% {proto.upper()} ({count:,} flows)")
    
    results['protocol_distribution'] = ", ".join(protocol_breakdown)
    
    port_dist = normal_data['port'].value_counts()
    port_breakdown = []
    port_names = {443: 'HTTPS', 53: 'DNS', 80: 'HTTP', 22: 'SSH', 21: 'FTP'}
    
    for port, count in port_dist.head(5).items():
        port_name = port_names.get(port, f"Port {port}")
        port_breakdown.append(f"{port_name} ({port}) with {count:,} flows")
    
    results['port_usage'] = ", ".join(port_breakdown)
    
    proto_port_combo = normal_data['proto'].astype(str) + ':' + normal_data['port'].astype(str)
    distribution = proto_port_combo.value_counts()
    
    top_combinations = []
    for i, (combo, count) in enumerate(distribution.head(10).items()):
        percentage = (count / distribution.sum()) * 100
        top_combinations.append(f"{percentage:.1f}% {combo} ({count:,} flows)")
    
    results['protocol_port_df'] = pd.DataFrame({
        'Protocol:Port': distribution.index,
        'Flow Count': distribution.values,
        'Percentage': (distribution.values / distribution.sum()) * 100
    })
    
    # 2. Normal Upload/Download Ratio per device
    device_stats = normal_data.groupby('src_ip').agg({
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    device_stats['up_down_ratio'] = device_stats['up_bytes'] / (device_stats['down_bytes'] + 1e-6)
    filtered_ratios = device_stats[device_stats['up_down_ratio'] <= 100]['up_down_ratio']
    
    ratio_mean = filtered_ratios.mean()
    ratio_median = filtered_ratios.median()
    ratio_std = filtered_ratios.std()
    ratio_min = filtered_ratios.quantile(0.05)
    ratio_max = filtered_ratios.quantile(0.95)
    
    results['upload_download_ratio'] = f"{ratio_min:.3f} - {ratio_max:.3f} (Mean: {ratio_mean:.3f}, Median: {ratio_median:.3f}, Std: {ratio_std:.3f})"
    results['ratio_stats'] = {
        'mean': ratio_mean,
        'median': ratio_median,
        'std': ratio_std,
        'min': ratio_min,
        'max': ratio_max
    }
    
    # 3. Normal HTTPS/DNS Ratio per device
    https_dns_stats = normal_data[normal_data['port'].isin([443, 53])].groupby(['src_ip', 'port']).size().unstack(fill_value=0)
    
    if 443 in https_dns_stats.columns and 53 in https_dns_stats.columns:
        devices_with_both = https_dns_stats[(https_dns_stats[443] > 0) & (https_dns_stats[53] > 0)]
        
        if len(devices_with_both) > 0:
            https_dns_ratios = devices_with_both[443] / devices_with_both[53]
            
            https_dns_mean = https_dns_ratios.mean()
            https_dns_median = https_dns_ratios.median()
            https_dns_std = https_dns_ratios.std()
            https_dns_min = https_dns_ratios.quantile(0.05)
            https_dns_max = https_dns_ratios.quantile(0.95)
            
            results['https_dns_ratio'] = f"{https_dns_min:.3f} - {https_dns_max:.3f} (Mean: {https_dns_mean:.3f}, Median: {https_dns_median:.3f}, Std: {https_dns_std:.3f})"
            results['https_dns_stats'] = {
                'mean': https_dns_mean,
                'median': https_dns_median,
                'std': https_dns_std,
                'min': https_dns_min,
                'max': https_dns_max
            }
        else:
            results['https_dns_ratio'] = "No devices with both HTTPS and DNS traffic"
            results['https_dns_stats'] = None
    else:
        results['https_dns_ratio'] = "HTTPS or DNS traffic not found"
        results['https_dns_stats'] = None
    
    # 4. Upload/Download Distribution (overall)
    total_upload = normal_data['up_bytes'].sum()
    total_download = normal_data['down_bytes'].sum()
    total_traffic = total_upload + total_download
    
    upload_percentage = (total_upload / total_traffic) * 100
    download_percentage = (total_download / total_traffic) * 100
    
    upload_gb = total_upload / (1024**3)
    download_gb = total_download / (1024**3)
    
    results['upload_download_distribution'] = f"{upload_percentage:.1f}% Upload ({upload_gb:.1f} GB), {download_percentage:.1f}% Download ({download_gb:.1f} GB)"
    results['traffic_stats'] = {
        'total_upload_bytes': total_upload,
        'total_download_bytes': total_download,
        'upload_gb': upload_gb,
        'download_gb': download_gb,
        'upload_percentage': upload_percentage,
        'download_percentage': download_percentage
    }
    
    return results

results = analyze_protocol_port_distribution(normal_data)
print("Protocol Distribution:", results['protocol_distribution'])
print("Port Usage:", results['port_usage'])
print("Normal Upload/Download Ratio:", results['upload_download_ratio'])
print("Normal HTTPS/DNS Ratio:", results['https_dns_ratio'])
print("Upload/Download Distribution:", results['upload_download_distribution'])

## 1.3 Geographical Analysis of External Connections

In [ ]:
def analyze_country_traffic(data, reader):
    """
    Analyze traffic by destination country with flow counts, percentages, and ratios.
    
    Parameters:
    -----------
    data : pandas.DataFrame
        Network data containing 'dst_ip', 'up_bytes', 'down_bytes' columns
    reader : geoip2.database.Reader
        Pre-initialized GeoIP2 reader object
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with Country, Percentage_Flows, and Ratio columns, ordered by flow count
    """
    import pandas as pd
    import geoip2.database
    import ipaddress
    
    # Filter for external destinations only (non-private IPs)
    def is_external_ip(ip):
        try:
            ip_obj = ipaddress.ip_address(ip)
            return not ip_obj.is_private
        except:
            return False
    
    # Filter external traffic
    external_data = data[data['dst_ip'].apply(is_external_ip)].copy()
    
    if len(external_data) == 0:
        return pd.DataFrame(columns=['Country', 'Percentage Flows', 'Ratio'])
    
    # Function to get country code
    def get_country(ip):
        try:
            response = reader.country(ip)
            return response.country.iso_code
        except:
            return 'Unknown'
    
    # Add country information
    external_data['country'] = external_data['dst_ip'].apply(get_country)
    
    # Group by country and calculate metrics
    country_stats = external_data.groupby('country').agg({
        'dst_ip': 'count',  # Flow count
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    country_stats.columns = ['Country', 'Flow_Count', 'Total_Upload', 'Total_Download']
    
    # Calculate percentage of flows
    total_flows = country_stats['Flow_Count'].sum()
    country_stats['Percentage Flows'] = (country_stats['Flow_Count'] / total_flows * 100).round(1)
    
    # Calculate upload/download ratio
    country_stats['Ratio'] = (country_stats['Total_Upload'] / 
                             (country_stats['Total_Download'] + 1e-6)).round(3)
    
    # Sort by flow count (descending)
    country_stats = country_stats.sort_values('Flow_Count', ascending=False)
    
    # Select and reorder final columns for clean table display
    result = country_stats[['Country', 'Percentage Flows', 'Ratio']].reset_index(drop=True)
    
    # Print summary to terminal
    print(f"\nCountry Traffic Analysis:")
    print(f"Total countries contacted: {len(result)}")
    print(f"Countries contacted: {', '.join(sorted(result['Country'].tolist()))}")
    print()
    
    return result

# Example usage:
country_df = analyze_country_traffic(normal_data, geoCC)
country_df.head(10)

# 2. Defining SIEM Rules

In this chapter, we define SIEM rules based on normal traffic patterns to detect network anomalies that may indicate attacks. We focus on five key areas: internal botnet activity through unusual device communication, data exfiltration via HTTPS and DNS channels, command-and-control communications using DNS traffic, connections to suspicious external destinations, and abnormal external user behavior when accessing corporate services. Each rule uses statistical thresholds derived from baseline data to minimize false positives while catching real threats.

## 2.1 Rule 1: Detecting Internal Botnet Activity

In [ ]:
def analyze_internal_services(normal_data):
    """
    Analyze internal services by identifying private destination IPs and their characteristics.
    
    Parameters:
    -----------
    normal_data : pandas.DataFrame
        The baseline network data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with Server IP, Flow Count, and Protocols columns
    """
    n_private_dst_data = normal_data[
        normal_data["dst_ip"].apply(lambda x: ipaddress.ip_address(x).is_private)
    ]
    
    n_private_dst_data_with_combo = n_private_dst_data.copy()
    n_private_dst_data_with_combo["proto_port"] = (
        n_private_dst_data_with_combo["proto"]
        + ":"
        + n_private_dst_data_with_combo["port"].astype(str)
    )
    
    private_dst_summary = (
        n_private_dst_data_with_combo.groupby("dst_ip")
        .agg({
            "proto_port": lambda x: sorted(set(x)),
            "dst_ip": "size"
        })
        .rename(columns={
            "proto_port": "Protocols", 
            "dst_ip": "Flow Count"
        })
        .sort_values("Flow Count", ascending=False)
    )
    
    services_df = private_dst_summary.reset_index()
    services_df = services_df.rename(columns={"dst_ip": "Server IP"})
    
    services_df["Protocols"] = services_df["Protocols"].apply(lambda x: ", ".join(x))
     
    return services_df

def calculate_internal_botnet_baseline(df):
    """
    Calculate baseline for internal botnet detection using normal traffic data.
    
    Parameters:
    df (pd.DataFrame): Normal traffic data with columns: timestamp, src_ip, dst_ip, proto, port, up_bytes, down_bytes
    
    Returns:
    dict: Baseline statistics and thresholds
    """
    
    # Step 1: Filter internal-to-internal communications using ipaddress.is_private
    is_internal_ip = lambda x: ipaddress.ip_address(x).is_private
    
    # Filter for internal-to-internal communications
    internal_mask = df['src_ip'].apply(is_internal_ip) & df['dst_ip'].apply(is_internal_ip)
    internal_flows = df[internal_mask].copy()
    
    print(f"Total flows in dataset: {len(df):,}")
    print(f"Internal-to-internal flows: {len(internal_flows):,}")
    
    # Step 2: Group by IP pairs and count flows
    ip_pair_counts = internal_flows.groupby(['src_ip', 'dst_ip']).size().reset_index(name='flow_count')
    
    # Step 3: Calculate baseline statistics
    baseline_stats = {
        'total_ip_pairs': len(ip_pair_counts),
        'mean_flows_per_pair': ip_pair_counts['flow_count'].mean(),
        'median_flows_per_pair': ip_pair_counts['flow_count'].median(),
        'std_flows_per_pair': ip_pair_counts['flow_count'].std(),
        'min_flows_per_pair': ip_pair_counts['flow_count'].min(),
        'max_flows_per_pair': ip_pair_counts['flow_count'].max(),
        'q25_flows_per_pair': ip_pair_counts['flow_count'].quantile(0.25),
        'q75_flows_per_pair': ip_pair_counts['flow_count'].quantile(0.75)
    }
    
    # Step 4: Set thresholds based on baseline
    baseline_avg = baseline_stats['mean_flows_per_pair']
    
    thresholds = {
        'baseline_average': baseline_avg,
        'high_threshold': baseline_avg * 10,      # 10x baseline
        'critical_threshold': baseline_avg * 25,  # 25x baseline
        'moderate_threshold': baseline_avg * 5    # 5x baseline
    }
    
    # Print baseline results
    print("\n=== BASELINE STATISTICS ===")
    print(f"Total IP pairs analyzed: {baseline_stats['total_ip_pairs']:,}")
    print(f"Average flows per IP pair: {baseline_avg:.1f}")
    print(f"Median flows per IP pair: {baseline_stats['median_flows_per_pair']:.1f}")
    print(f"Standard deviation: {baseline_stats['std_flows_per_pair']:.1f}")
    print(f"Range: {baseline_stats['min_flows_per_pair']:.0f} - {baseline_stats['max_flows_per_pair']:.0f}")
    
    print(f"\n=== DETECTION THRESHOLDS ===")
    print(f"Moderate threshold (5x):  {thresholds['moderate_threshold']:.1f} flows")
    print(f"High threshold (10x):     {thresholds['high_threshold']:.1f} flows")
    print(f"Critical threshold (25x): {thresholds['critical_threshold']:.1f} flows")
    
    return baseline_stats, thresholds, ip_pair_counts

def detect_internal_botnet(test_df, thresholds, server_ips):
    """
    Apply botnet detection rules to test data, including lateral movement detection.
    
    Parameters:
    test_df (pd.DataFrame): Test data to analyze
    thresholds (dict): Detection thresholds from baseline calculation
    server_ips (set): Set of internal server IP addresses
    
    Returns:
    pd.DataFrame: Detected anomalous IP pairs
    """
    
    # Filter internal-to-internal communications using ipaddress.is_private
    is_internal_ip = lambda x: ipaddress.ip_address(x).is_private
    
    internal_mask = test_df['src_ip'].apply(is_internal_ip) & test_df['dst_ip'].apply(is_internal_ip)
    internal_flows = test_df[internal_mask].copy()
    
    # Group by IP pairs and count flows
    ip_pair_counts = internal_flows.groupby(['src_ip', 'dst_ip']).size().reset_index(name='flow_count')
    
    # Add lateral movement detection
    ip_pair_counts['is_lateral_movement'] = ~ip_pair_counts['dst_ip'].isin(server_ips)
    
    # Apply thresholds with lateral movement consideration
    critical_detections = ip_pair_counts[
        (ip_pair_counts['flow_count'] >= thresholds['critical_threshold']) |
        (ip_pair_counts['is_lateral_movement'] == True)  # Any lateral movement is critical
    ].copy()
    
    high_detections = ip_pair_counts[
        (ip_pair_counts['flow_count'] >= thresholds['high_threshold']) & 
        (ip_pair_counts['flow_count'] < thresholds['critical_threshold']) &
        (ip_pair_counts['is_lateral_movement'] == False)  # Only server communications
    ].copy()
    
    moderate_detections = ip_pair_counts[
        (ip_pair_counts['flow_count'] >= thresholds['moderate_threshold']) & 
        (ip_pair_counts['flow_count'] < thresholds['high_threshold']) &
        (ip_pair_counts['is_lateral_movement'] == False)  # Only server communications
    ].copy()
    
    # Add severity labels and detection reasons
    critical_detections['severity'] = 'CRITICAL'
    critical_detections['detection_reason'] = critical_detections.apply(
        lambda row: 'LATERAL_MOVEMENT' if row['is_lateral_movement'] else 'HIGH_VOLUME', axis=1
    )
    
    high_detections['severity'] = 'HIGH'
    high_detections['detection_reason'] = 'HIGH_VOLUME'
    
    moderate_detections['severity'] = 'MODERATE'
    moderate_detections['detection_reason'] = 'MODERATE_VOLUME'
    
    # Combine results
    all_detections = pd.concat([critical_detections, high_detections, moderate_detections], ignore_index=True)
    
    # Calculate severity multiplier
    if not all_detections.empty:
        all_detections['baseline_multiplier'] = all_detections['flow_count'] / thresholds['baseline_average']
    
    # Sort by severity (lateral movement first, then by flow count)
    all_detections = all_detections.sort_values(
        ['detection_reason', 'flow_count'], 
        ascending=[False, False]  # LATERAL_MOVEMENT first, then highest flow count
    )
    
    # Separate lateral movement and volume-based detections for reporting
    lateral_movement_detections = all_detections[all_detections['detection_reason'] == 'LATERAL_MOVEMENT']
    volume_detections = all_detections[all_detections['detection_reason'] != 'LATERAL_MOVEMENT']
    
    # Print detection results
    print(f"\n=== BOTNET DETECTION RESULTS ===")
    print(f"Lateral movement detections: {len(lateral_movement_detections)}")
    print(f"Volume-based detections:")
    print(f"  - Critical: {len(critical_detections[critical_detections['detection_reason'] == 'HIGH_VOLUME'])}")
    print(f"  - High: {len(high_detections)}")
    print(f"  - Moderate: {len(moderate_detections)}")
    print(f"Total anomalous IP pairs: {len(all_detections)}")
    
    # Report lateral movement cases
    if not lateral_movement_detections.empty:
        print(f"\n=== LATERAL MOVEMENT DETECTIONS (CRITICAL) ===")
        print(f"Total lateral movement cases: {len(lateral_movement_detections)}")
        print("Source IP -> Destination IP (Non-Server) | Flow Count | Baseline Multiplier")
        print("-" * 80)
        
        # Show ALL lateral movement detections, not just top 10
        for _, row in lateral_movement_detections.iterrows():
            print(f"{row['src_ip']} -> {row['dst_ip']} | {row['flow_count']} | {row['baseline_multiplier']:.1f}x")
        
        # Additional analysis: show unique source IPs and their targets
        print(f"\n=== LATERAL MOVEMENT SUMMARY ===")
        lateral_summary = lateral_movement_detections.groupby('src_ip').agg({
            'dst_ip': lambda x: list(x),
            'flow_count': 'sum'
        }).reset_index()
        lateral_summary['target_count'] = lateral_summary['dst_ip'].apply(len)
        lateral_summary = lateral_summary.sort_values('target_count', ascending=False)
        
        print("Source IP | Targets | Total Flows | Target IPs")
        print("-" * 60)
        for _, row in lateral_summary.iterrows():
            targets_str = ', '.join(row['dst_ip'])
            print(f"{row['src_ip']} | {row['target_count']} | {row['flow_count']} | {targets_str}")
    
    # Report volume-based detections
    if not volume_detections.empty:
        print(f"\n=== VOLUME-BASED DETECTIONS ===")
        print("Severe Cases:")
        print(volume_detections[['src_ip', 'dst_ip', 'flow_count', 'severity', 'baseline_multiplier', 'detection_reason']].to_string(index=False))
    
    return all_detections

server_ips = analyze_internal_services(normal_data)['Server IP'].unique()
baseline_stats, thresholds, ip_pairs = calculate_internal_botnet_baseline(normal_data)
detections = detect_internal_botnet(anomalous_data, thresholds, server_ips)
    
if not detections.empty:
    compromised_ips = pd.concat([detections['src_ip'], detections['dst_ip']]).unique()
    print(f"\nUnique compromised IPs: {len(compromised_ips)}")
    print(compromised_ips.tolist())
    
    lateral_movement_ips = detections[detections['detection_reason'] == 'LATERAL_MOVEMENT']['src_ip'].unique()
    if len(lateral_movement_ips) > 0:
        print(f"\nLateral Movement Source IPs (HIGH PRIORITY): {len(lateral_movement_ips)}")
        print(lateral_movement_ips.tolist())

## 2.2 Rule 2: Detecting HTTPS Data Exfiltration

In [ ]:
# %% [markdown]
# ## 2.2 Rule 2: HTTPS Data Exfiltration Detection

# %%
def calculate_https_exfiltration_baseline(df):
    """
    Calculate baseline for HTTPS data exfiltration detection using normal traffic data.
    
    Parameters:
    df (pd.DataFrame): Normal traffic data with columns: timestamp, src_ip, dst_ip, proto, port, up_bytes, down_bytes
    
    Returns:
    tuple: (baseline_stats, thresholds)
    """
    
    # Step 1: Filter for HTTPS traffic only (port 443)
    https_traffic = df[df['port'] == 443].copy()
    
    print(f"Total flows in dataset: {len(df):,}")
    print(f"HTTPS flows (port 443): {len(https_traffic):,}")
    
    if len(https_traffic) == 0:
        print("No HTTPS traffic found in dataset")
        return None, None
    
    # Step 2: Group by source IP and calculate upload/download ratios
    device_stats = https_traffic.groupby('src_ip').agg({
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    # Calculate upload/download ratio with small epsilon to avoid division by zero
    device_stats['up_down_ratio'] = device_stats['up_bytes'] / (device_stats['down_bytes'] + 1e-6)
    
    # Filter out extreme outliers (ratios > 100) for baseline calculation
    filtered_stats = device_stats[device_stats['up_down_ratio'] <= 100].copy()
    
    print(f"Devices with HTTPS traffic: {len(device_stats)}")
    print(f"Devices used for baseline (ratio ≤ 100): {len(filtered_stats)}")
    
    # Step 3: Calculate baseline statistics
    ratios = filtered_stats['up_down_ratio']
    
    baseline_stats = {
        'total_devices': len(device_stats),
        'baseline_devices': len(filtered_stats),
        'mean_ratio': ratios.mean(),
        'median_ratio': ratios.median(),
        'std_ratio': ratios.std(),
        'min_ratio': ratios.min(),
        'max_ratio': ratios.max(),
        'q5_ratio': ratios.quantile(0.05),
        'q95_ratio': ratios.quantile(0.95),
        'q25_ratio': ratios.quantile(0.25),
        'q75_ratio': ratios.quantile(0.75)
    }
    
    # Step 4: Set detection thresholds based on baseline
    # Normal range: 5th to 95th percentile
    normal_min = baseline_stats['q5_ratio']
    normal_max = baseline_stats['q95_ratio']
    boundary_tolerance = 0.005
    
    # Calculate thresholds based on statistical analysis
    normal_range = [normal_min, normal_max]
    
    # High exfiltration: 5x normal maximum
    high_threshold = normal_max * 5
    
    # Critical exfiltration: 10x normal maximum  
    critical_threshold = normal_max * 10
    
    thresholds = {
        'normal_range': normal_range,
        'boundary_tolerance': boundary_tolerance,
        'high_threshold': high_threshold,
        'critical_threshold': critical_threshold,
        'normal_min': normal_min,
        'normal_max': normal_max
    }
    
    # Print baseline results
    print("\n=== HTTPS EXFILTRATION BASELINE STATISTICS ===")
    print(f"Mean upload/download ratio: {baseline_stats['mean_ratio']:.3f}")
    print(f"Median upload/download ratio: {baseline_stats['median_ratio']:.3f}")
    print(f"Standard deviation: {baseline_stats['std_ratio']:.3f}")
    print(f"Normal range (5th-95th percentile): {normal_min:.3f} - {normal_max:.3f}")
    print(f"Interquartile range: {baseline_stats['q25_ratio']:.3f} - {baseline_stats['q75_ratio']:.3f}")
    
    print(f"\n=== DETECTION THRESHOLDS ===")
    print(f"Normal Range: [{normal_min:.3f}, {normal_max:.3f}]")
    print(f"Boundary Tolerance: ±{boundary_tolerance:.3f}")
    print(f"High Exfiltration Threshold: > {high_threshold:.3f}")
    print(f"Critical Exfiltration Threshold: > {critical_threshold:.3f}")
    
    return baseline_stats, thresholds

def detect_https_exfiltration(test_df, thresholds):
    """
    Apply HTTPS data exfiltration detection rules to test data.
    
    Parameters:
    test_df (pd.DataFrame): Test data to analyze
    thresholds (dict): Detection thresholds from baseline calculation
    
    Returns:
    pd.DataFrame: Detected anomalous devices with exfiltration patterns
    """
    
    if thresholds is None:
        print("No thresholds provided - cannot perform detection")
        return pd.DataFrame()
    
    # Step 1: Filter for HTTPS traffic only (port 443)
    https_traffic = test_df[test_df['port'] == 443].copy()
    
    if len(https_traffic) == 0:
        print("No HTTPS traffic found in test dataset")
        return pd.DataFrame()
    
    # Step 2: Group by source IP and calculate upload/download ratios
    device_stats = https_traffic.groupby('src_ip').agg({
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    # Calculate upload/download ratio
    device_stats['up_down_ratio'] = device_stats['up_bytes'] / (device_stats['down_bytes'] + 1e-6)
    
    # Convert bytes to MB for better readability
    device_stats['up_mb'] = device_stats['up_bytes'] / (1024 * 1024)
    device_stats['down_mb'] = device_stats['down_bytes'] / (1024 * 1024)
    
    # Step 3: Apply detection thresholds
    normal_min = thresholds['normal_min']
    normal_max = thresholds['normal_max']
    boundary_tolerance = thresholds['boundary_tolerance']
    high_threshold = thresholds['high_threshold']
    critical_threshold = thresholds['critical_threshold']
    
    # Identify anomalous devices
    detections = []
    
    for _, row in device_stats.iterrows():
        ratio = row['up_down_ratio']
        
        # Skip if within normal range (with tolerance)
        if (normal_min - boundary_tolerance) <= ratio <= (normal_max + boundary_tolerance):
            continue
        
        # Determine severity
        if ratio >= critical_threshold:
            severity = 'CRITICAL'
        elif ratio >= high_threshold:
            severity = 'HIGH'
        elif ratio > (normal_max + boundary_tolerance):
            severity = 'MODERATE'
        else:
            continue
        
        # Calculate percentage above normal
        if ratio > normal_max:
            percentage_above_normal = ((ratio - normal_max) / normal_max) * 100
        else:
            percentage_above_normal = 0
        
        detections.append({
            'src_ip': row['src_ip'],
            'up_down_ratio': ratio,
            'up_mb': row['up_mb'],
            'down_mb': row['down_mb'],
            'severity': severity,
            'percentage_above_normal': percentage_above_normal
        })
    
    # Convert to DataFrame
    if not detections:
        print("No HTTPS exfiltration anomalies detected")
        return pd.DataFrame()
    
    detection_df = pd.DataFrame(detections)
    
    # Sort by severity and ratio
    severity_order = {'CRITICAL': 4, 'HIGH': 3, 'MODERATE': 2}
    detection_df['severity_rank'] = detection_df['severity'].map(severity_order)
    detection_df = detection_df.sort_values(['severity_rank', 'up_down_ratio'], ascending=[False, False])
    
    # Step 4: Print detection results
    print(f"\n=== HTTPS EXFILTRATION DETECTION RESULTS ===")
    print(f"Total devices with HTTPS traffic: {len(device_stats)}")
    print(f"Devices with anomalous upload/download ratios: {len(detection_df)}")
    
    # Group by severity
    severity_counts = detection_df['severity'].value_counts()
    for severity in ['CRITICAL', 'HIGH', 'MODERATE']:
        if severity in severity_counts:
            print(f"  - {severity}: {severity_counts[severity]}")
    
    if len(detection_df) > 0:
        print(f"\n=== DETAILED DETECTION RESULTS ===")
        print("IP Address | Ratio | Upload (MB) | Download (MB) | Severity | % Above Normal")
        print("-" * 85)
        
        for _, row in detection_df.iterrows():
            print(f"{row['src_ip']} | {row['up_down_ratio']:.3f} | {row['up_mb']:.1f} | {row['down_mb']:.1f} | {row['severity']} | {row['percentage_above_normal']:.1f}%")
        
        # Highlight most severe cases
        critical_cases = detection_df[detection_df['severity'] == 'CRITICAL']
        if len(critical_cases) > 0:
            print(f"\n=== CRITICAL EXFILTRATION CASES ===")
            for _, row in critical_cases.iterrows():
                print(f"{row['src_ip']}: {row['percentage_above_normal']:.0f}% above normal ratio")
                print(f"  └─ Uploaded {row['up_mb']:.1f} MB vs Downloaded {row['down_mb']:.1f} MB")
    
    return detection_df.drop(columns=['severity_rank'])

# Execute the HTTPS exfiltration detection
print("\n" + "="*60)
print("RULE 2: HTTPS DATA EXFILTRATION DETECTION")
print("="*60)

baseline_stats_https, thresholds_https = calculate_https_exfiltration_baseline(normal_data)

if thresholds_https is not None:
    https_detections = detect_https_exfiltration(anomalous_data, thresholds_https)
    
    if not https_detections.empty:
        print(f"\n=== SUMMARY ===")
        exfiltration_ips = https_detections['src_ip'].unique()
        print(f"Unique IPs with HTTPS exfiltration patterns: {len(exfiltration_ips)}")
        print(f"IPs: {exfiltration_ips.tolist()}")
        
        # Show top exfiltration cases
        critical_high = https_detections[https_detections['severity'].isin(['CRITICAL', 'HIGH'])]
        if len(critical_high) > 0:
            print(f"\nHigh-priority exfiltration IPs: {critical_high['src_ip'].tolist()}")
    else:
        print(f"\n=== SUMMARY ===")
        print("No HTTPS data exfiltration patterns detected in the test dataset.")

## 2.3 Rule 3: Detecting DNS Data Exfiltration

In [ ]:
def calculate_dns_exfiltration_baseline(df):
    """
    Calculate baseline for DNS data exfiltration detection using normal traffic data.
    
    Parameters:
    df (pd.DataFrame): Normal traffic data with columns: timestamp, src_ip, dst_ip, proto, port, up_bytes, down_bytes
    
    Returns:
    tuple: (baseline_stats, thresholds)
    """
    
    # Step 1: Filter for HTTPS and DNS traffic
    https_dns_traffic = df[df['port'].isin([443, 53])].copy()
    
    print(f"Total flows in dataset: {len(df):,}")
    print(f"HTTPS + DNS flows: {len(https_dns_traffic):,}")
    
    if len(https_dns_traffic) == 0:
        print("No HTTPS or DNS traffic found in dataset")
        return None, None
    
    # Step 2: Group by source IP and protocol/port to count flows
    device_protocol_stats = https_dns_traffic.groupby(['src_ip', 'port']).size().unstack(fill_value=0)
    
    # Ensure both columns exist
    if 443 not in device_protocol_stats.columns:
        device_protocol_stats[443] = 0
    if 53 not in device_protocol_stats.columns:
        device_protocol_stats[53] = 0
    
    # Step 3: Filter devices with both HTTPS and DNS traffic
    devices_with_both = device_protocol_stats[(device_protocol_stats[443] > 0) & (device_protocol_stats[53] > 0)].copy()
    
    print(f"Devices with HTTPS traffic: {len(device_protocol_stats[device_protocol_stats[443] > 0])}")
    print(f"Devices with DNS traffic: {len(device_protocol_stats[device_protocol_stats[53] > 0])}")
    print(f"Devices with both HTTPS and DNS: {len(devices_with_both)}")
    
    if len(devices_with_both) == 0:
        print("No devices with both HTTPS and DNS traffic found")
        return None, None
    
    # Step 4: Calculate HTTPS/DNS ratios
    devices_with_both['https_dns_ratio'] = devices_with_both[443] / devices_with_both[53]
    
    # Step 5: Calculate baseline statistics
    ratios = devices_with_both['https_dns_ratio']
    
    baseline_stats = {
        'total_devices': len(devices_with_both),
        'mean_ratio': ratios.mean(),
        'median_ratio': ratios.median(),
        'std_ratio': ratios.std(),
        'min_ratio': ratios.min(),
        'max_ratio': ratios.max(),
        'q5_ratio': ratios.quantile(0.05),
        'q95_ratio': ratios.quantile(0.95),
        'q25_ratio': ratios.quantile(0.25),
        'q75_ratio': ratios.quantile(0.75)
    }
    
    # Step 6: Set detection thresholds
    normal_min = baseline_stats['q5_ratio']
    normal_max = baseline_stats['q95_ratio']
    mean_ratio = baseline_stats['mean_ratio']
    std_ratio = baseline_stats['std_ratio']
    
    # DNS Tunneling thresholds (too much DNS = low HTTPS/DNS ratio)
    critical_dns_tunneling = normal_min * 0.3  # 70% below normal minimum
    high_dns_tunneling = normal_min * 0.7      # 30% below normal minimum
    
    # DNS Suppression thresholds (too little DNS = high HTTPS/DNS ratio)
    high_dns_suppression = normal_max * 1.5    # 50% above normal maximum
    critical_dns_suppression = normal_max * 2.0 # 100% above normal maximum
    
    thresholds = {
        'normal_min': normal_min,
        'normal_max': normal_max,
        'critical_dns_tunneling': critical_dns_tunneling,
        'high_dns_tunneling': high_dns_tunneling,
        'high_dns_suppression': high_dns_suppression,
        'critical_dns_suppression': critical_dns_suppression
    }
    
    # Print baseline results
    print("\n=== DNS EXFILTRATION BASELINE STATISTICS ===")
    print(f"Mean HTTPS/DNS ratio: {baseline_stats['mean_ratio']:.3f}")
    print(f"Median HTTPS/DNS ratio: {baseline_stats['median_ratio']:.3f}")
    print(f"Standard deviation: {baseline_stats['std_ratio']:.3f}")
    print(f"Normal range (5th-95th percentile): {normal_min:.3f} - {normal_max:.3f}")
    print(f"Interquartile range: {baseline_stats['q25_ratio']:.3f} - {baseline_stats['q75_ratio']:.3f}")
    
    print(f"\n=== DETECTION THRESHOLDS ===")
    print(f"Normal HTTPS/DNS Ratio Range: {normal_min:.3f} - {normal_max:.3f}")
    print(f"Critical DNS Tunneling: < {critical_dns_tunneling:.3f} (excessive DNS)")
    print(f"High DNS Tunneling: < {high_dns_tunneling:.3f} (suspicious DNS)")
    print(f"High DNS Suppression: > {high_dns_suppression:.3f} (low DNS usage)")
    print(f"Critical DNS Suppression: > {critical_dns_suppression:.3f} (very low DNS)")
    
    return baseline_stats, thresholds

def detect_dns_exfiltration(test_df, thresholds):
    """
    Apply DNS data exfiltration detection rules to test data.
    
    Parameters:
    test_df (pd.DataFrame): Test data to analyze
    thresholds (dict): Detection thresholds from baseline calculation
    
    Returns:
    pd.DataFrame: Detected anomalous devices with DNS exfiltration patterns
    """
    
    if thresholds is None:
        print("No thresholds provided - cannot perform detection")
        return pd.DataFrame()
    
    # Step 1: Filter for HTTPS and DNS traffic
    https_dns_traffic = test_df[test_df['port'].isin([443, 53])].copy()
    
    if len(https_dns_traffic) == 0:
        print("No HTTPS or DNS traffic found in test dataset")
        return pd.DataFrame()
    
    # Step 2: Group by source IP and protocol/port to count flows
    device_protocol_stats = https_dns_traffic.groupby(['src_ip', 'port']).size().unstack(fill_value=0)
    
    # Ensure both columns exist
    if 443 not in device_protocol_stats.columns:
        device_protocol_stats[443] = 0
    if 53 not in device_protocol_stats.columns:
        device_protocol_stats[53] = 0
    
    # Step 3: Filter devices with both HTTPS and DNS traffic
    devices_with_both = device_protocol_stats[(device_protocol_stats[443] > 0) & (device_protocol_stats[53] > 0)].copy()
    
    if len(devices_with_both) == 0:
        print("No devices with both HTTPS and DNS traffic found in test dataset")
        return pd.DataFrame()
    
    # Step 4: Calculate HTTPS/DNS ratios
    devices_with_both['https_dns_ratio'] = devices_with_both[443] / devices_with_both[53]
    devices_with_both = devices_with_both.reset_index()
    
    # Step 5: Apply detection thresholds
    normal_min = thresholds['normal_min']
    normal_max = thresholds['normal_max']
    critical_dns_tunneling = thresholds['critical_dns_tunneling']
    high_dns_tunneling = thresholds['high_dns_tunneling']
    high_dns_suppression = thresholds['high_dns_suppression']
    critical_dns_suppression = thresholds['critical_dns_suppression']
    
    # Identify anomalous devices
    detections = []
    
    for _, row in devices_with_both.iterrows():
        ratio = row['https_dns_ratio']
        https_flows = row[443]
        dns_flows = row[53]
        
        # Skip if within normal range
        if normal_min <= ratio <= normal_max:
            continue
        
        # Determine severity and type
        if ratio < critical_dns_tunneling:
            severity = 'CRITICAL'
            anomaly_type = 'DNS_TUNNELING'
            description = 'Excessive DNS queries'
        elif ratio < high_dns_tunneling:
            severity = 'HIGH'
            anomaly_type = 'DNS_TUNNELING'
            description = 'Suspicious DNS activity'
        elif ratio > critical_dns_suppression:
            severity = 'CRITICAL'
            anomaly_type = 'DNS_SUPPRESSION'
            description = 'Very low DNS usage'
        elif ratio > high_dns_suppression:
            severity = 'HIGH'
            anomaly_type = 'DNS_SUPPRESSION'
            description = 'Low DNS usage'
        else:
            continue
        
        # Calculate percentage deviation from normal
        if ratio < normal_min:
            percentage_deviation = ((normal_min - ratio) / normal_min) * 100
            deviation_direction = 'below normal minimum'
        else:
            percentage_deviation = ((ratio - normal_max) / normal_max) * 100
            deviation_direction = 'above normal maximum'
        
        detections.append({
            'src_ip': row['src_ip'],
            'https_dns_ratio': ratio,
            'https_flows': https_flows,
            'dns_flows': dns_flows,
            'severity': severity,
            'anomaly_type': anomaly_type,
            'description': description,
            'percentage_deviation': percentage_deviation,
            'deviation_direction': deviation_direction
        })
    
    # Convert to DataFrame
    if not detections:
        print("No DNS exfiltration anomalies detected")
        return pd.DataFrame()
    
    detection_df = pd.DataFrame(detections)
    
    # Sort by severity and ratio (tunneling first, then suppression)
    severity_order = {'CRITICAL': 4, 'HIGH': 3}
    type_order = {'DNS_TUNNELING': 2, 'DNS_SUPPRESSION': 1}
    detection_df['severity_rank'] = detection_df['severity'].map(severity_order)
    detection_df['type_rank'] = detection_df['anomaly_type'].map(type_order)
    detection_df = detection_df.sort_values(['severity_rank', 'type_rank', 'https_dns_ratio'], ascending=[False, False, True])
    
    # Step 6: Print detection results
    print(f"\n=== DNS EXFILTRATION DETECTION RESULTS ===")
    print(f"Total devices with HTTPS and DNS traffic: {len(devices_with_both)}")
    print(f"Devices with anomalous HTTPS/DNS ratios: {len(detection_df)}")
    
    # Group by severity and type
    tunneling_detections = detection_df[detection_df['anomaly_type'] == 'DNS_TUNNELING']
    suppression_detections = detection_df[detection_df['anomaly_type'] == 'DNS_SUPPRESSION']
    
    if len(tunneling_detections) > 0:
        tunneling_counts = tunneling_detections['severity'].value_counts()
        print(f"DNS Tunneling detections:")
        for severity in ['CRITICAL', 'HIGH']:
            if severity in tunneling_counts:
                print(f"  - {severity}: {tunneling_counts[severity]}")
    
    if len(suppression_detections) > 0:
        suppression_counts = suppression_detections['severity'].value_counts()
        print(f"DNS Suppression detections:")
        for severity in ['CRITICAL', 'HIGH']:
            if severity in suppression_counts:
                print(f"  - {severity}: {suppression_counts[severity]}")
    
    if len(detection_df) > 0:
        print(f"\n=== DETAILED DETECTION RESULTS ===")
        print("IP Address | Ratio | HTTPS | DNS | Severity | Type | % Deviation")
        print("-" * 80)
        
        for _, row in detection_df.iterrows():
            print(f"{row['src_ip']} | {row['https_dns_ratio']:.3f} | {row['https_flows']} | {row['dns_flows']} | {row['severity']} | {row['anomaly_type']} | {row['percentage_deviation']:.1f}%")
        
        # Highlight most severe cases
        critical_cases = detection_df[detection_df['severity'] == 'CRITICAL']
        if len(critical_cases) > 0:
            print(f"\n=== CRITICAL DNS EXFILTRATION CASES ===")
            for _, row in critical_cases.iterrows():
                print(f"{row['src_ip']}: {row['description']}")
                print(f"  └─ HTTPS/DNS ratio: {row['https_dns_ratio']:.3f} ({row['percentage_deviation']:.1f}% {row['deviation_direction']})")
                print(f"  └─ {row['https_flows']} HTTPS flows, {row['dns_flows']} DNS flows")
    
    return detection_df.drop(columns=['severity_rank', 'type_rank'])

# Execute the DNS exfiltration detection
print("\n" + "="*60)
print("RULE 3: DNS DATA EXFILTRATION DETECTION")
print("="*60)

baseline_stats_dns, thresholds_dns = calculate_dns_exfiltration_baseline(normal_data)

if thresholds_dns is not None:
    dns_detections = detect_dns_exfiltration(anomalous_data, thresholds_dns)
    
    if not dns_detections.empty:
        print(f"\n=== SUMMARY ===")
        dns_exfiltration_ips = dns_detections['src_ip'].unique()
        print(f"Unique IPs with DNS exfiltration patterns: {len(dns_exfiltration_ips)}")
        print(f"IPs: {dns_exfiltration_ips.tolist()}")
        
        # Show tunneling and suppression cases separately
        tunneling_ips = dns_detections[dns_detections['anomaly_type'] == 'DNS_TUNNELING']['src_ip'].unique()
        suppression_ips = dns_detections[dns_detections['anomaly_type'] == 'DNS_SUPPRESSION']['src_ip'].unique()
        
        if len(tunneling_ips) > 0:
            print(f"\nDNS Tunneling IPs (excessive DNS): {tunneling_ips.tolist()}")
        if len(suppression_ips) > 0:
            print(f"DNS Suppression IPs (low DNS usage): {suppression_ips.tolist()}")
        
        # Show critical cases
        critical_cases = dns_detections[dns_detections['severity'] == 'CRITICAL']
        if len(critical_cases) > 0:
            print(f"\nCritical DNS anomaly IPs: {critical_cases['src_ip'].tolist()}")
    else:
        print(f"\n=== SUMMARY ===")
        print("No DNS data exfiltration patterns detected in the test dataset.")

## 2.4 Rule 4: DNS Command & Control (C&C) Activities Detection

In [ ]:
def calculate_dns_cc_baseline(df):
    """
    Calculate baseline for DNS C&C detection using normal traffic data.
    
    Parameters:
    df (pd.DataFrame): Normal traffic data with columns: timestamp, src_ip, dst_ip, proto, port, up_bytes, down_bytes
    
    Returns:
    tuple: (baseline_stats, thresholds)
    """
    
    # Step 1: Filter for DNS traffic only (port 53)
    dns_traffic = df[df['port'] == 53].copy()
    
    print(f"Total flows in dataset: {len(df):,}")
    print(f"DNS flows (port 53): {len(dns_traffic):,}")
    
    if len(dns_traffic) == 0:
        print("No DNS traffic found in dataset")
        return None, None
    
    # Step 2: Count DNS queries per device
    device_dns_counts = dns_traffic.groupby('src_ip').size().reset_index(name='dns_queries')
    
    print(f"Devices with DNS traffic: {len(device_dns_counts)}")
    
    # Step 3: Calculate baseline statistics
    dns_counts = device_dns_counts['dns_queries']
    
    baseline_stats = {
        'total_devices': len(device_dns_counts),
        'mean_queries': dns_counts.mean(),
        'median_queries': dns_counts.median(),
        'std_queries': dns_counts.std(),
        'min_queries': dns_counts.min(),
        'max_queries': dns_counts.max(),
        'q25_queries': dns_counts.quantile(0.25),
        'q75_queries': dns_counts.quantile(0.75),
        'q95_queries': dns_counts.quantile(0.95),
        'q99_queries': dns_counts.quantile(0.99)
    }
    
    # Step 4: Set conservative detection thresholds using statistical analysis
    mean_queries = baseline_stats['mean_queries']
    std_queries = baseline_stats['std_queries']
    max_queries = baseline_stats['max_queries']
    q95_queries = baseline_stats['q95_queries']
    
    # Conservative thresholds to minimize false positives
    # High: Range + 1 standard deviation (captures extreme outliers)
    high_threshold = max_queries + std_queries
    
    # Critical: 2x maximum normal (statistically impossible for legitimate users)
    critical_threshold = max_queries * 2
    
    # Alternative: Use 99th percentile + 2 standard deviations for very conservative detection
    conservative_threshold = q95_queries + (2 * std_queries)
    
    thresholds = {
        'mean_queries': mean_queries,
        'std_queries': std_queries,
        'max_queries': max_queries,
        'high_threshold': high_threshold,
        'critical_threshold': critical_threshold,
        'conservative_threshold': conservative_threshold
    }
    
    # Print baseline results
    print("\n=== DNS C&C BASELINE STATISTICS ===")
    print(f"Mean DNS queries per device: {baseline_stats['mean_queries']:.1f}")
    print(f"Median DNS queries per device: {baseline_stats['median_queries']:.1f}")
    print(f"Standard deviation: {baseline_stats['std_queries']:.1f}")
    print(f"Normal range: {baseline_stats['min_queries']:.0f} - {baseline_stats['max_queries']:.0f} queries")
    print(f"95th percentile: {baseline_stats['q95_queries']:.1f} queries")
    print(f"99th percentile: {baseline_stats['q99_queries']:.1f} queries")
    
    print(f"\n=== DETECTION THRESHOLDS ===")
    print(f"High Threshold: > {high_threshold:.1f} queries (max + 1 std dev)")
    print(f"Critical Threshold: > {critical_threshold:.1f} queries (2× max normal)")
    print(f"Conservative Threshold: > {conservative_threshold:.1f} queries (95th percentile + 2 std dev)")
    
    return baseline_stats, thresholds

def detect_dns_cc(test_df, thresholds):
    """
    Apply DNS C&C detection rules to test data.
    
    Parameters:
    test_df (pd.DataFrame): Test data to analyze
    thresholds (dict): Detection thresholds from baseline calculation
    
    Returns:
    pd.DataFrame: Detected anomalous devices with excessive DNS patterns
    """
    
    if thresholds is None:
        print("No thresholds provided - cannot perform detection")
        return pd.DataFrame()
    
    # Step 1: Filter for DNS traffic only (port 53)
    dns_traffic = test_df[test_df['port'] == 53].copy()
    
    if len(dns_traffic) == 0:
        print("No DNS traffic found in test dataset")
        return pd.DataFrame()
    
    # Step 2: Count DNS queries per device
    device_dns_counts = dns_traffic.groupby('src_ip').size().reset_index(name='dns_queries')
    
    # Step 3: Apply detection thresholds
    mean_queries = thresholds['mean_queries']
    std_queries = thresholds['std_queries']
    high_threshold = thresholds['high_threshold']
    critical_threshold = thresholds['critical_threshold']
    
    # Identify anomalous devices
    detections = []
    
    for _, row in device_dns_counts.iterrows():
        queries = row['dns_queries']
        
        # Determine severity
        if queries >= critical_threshold:
            severity = 'CRITICAL'
        elif queries >= high_threshold:
            severity = 'HIGH'
        else:
            continue
        
        # Calculate statistical measures
        std_above_mean = (queries - mean_queries) / std_queries if std_queries > 0 else 0
        percentage_above_max = ((queries - thresholds['max_queries']) / thresholds['max_queries']) * 100
        
        detections.append({
            'src_ip': row['src_ip'],
            'dns_queries': queries,
            'severity': severity,
            'std_above_mean': std_above_mean,
            'percentage_above_max': percentage_above_max
        })
    
    # Convert to DataFrame
    if not detections:
        print("No DNS C&C anomalies detected")
        return pd.DataFrame()
    
    detection_df = pd.DataFrame(detections)
    
    # Sort by severity and query count
    severity_order = {'CRITICAL': 4, 'HIGH': 3}
    detection_df['severity_rank'] = detection_df['severity'].map(severity_order)
    detection_df = detection_df.sort_values(['severity_rank', 'dns_queries'], ascending=[False, False])
    
    # Step 4: Print detection results
    print(f"\n=== DNS C&C DETECTION RESULTS ===")
    print(f"Total devices with DNS traffic: {len(device_dns_counts)}")
    print(f"Devices with excessive DNS queries: {len(detection_df)}")
    
    # Group by severity
    severity_counts = detection_df['severity'].value_counts()
    for severity in ['CRITICAL', 'HIGH']:
        if severity in severity_counts:
            print(f"  - {severity}: {severity_counts[severity]}")
    
    if len(detection_df) > 0:
        print(f"\n=== DETAILED DETECTION RESULTS ===")
        print("IP Address | DNS Queries | Std Above Mean | % Above Max | Severity")
        print("-" * 75)
        
        for _, row in detection_df.iterrows():
            print(f"{row['src_ip']} | {row['dns_queries']:,} | {row['std_above_mean']:.1f} std | {row['percentage_above_max']:.1f}% | {row['severity']}")
        
        # Highlight most severe cases
        critical_cases = detection_df[detection_df['severity'] == 'CRITICAL']
        if len(critical_cases) > 0:
            print(f"\n=== CRITICAL DNS C&C CASES ===")
            for _, row in critical_cases.iterrows():
                print(f"{row['src_ip']}: {row['dns_queries']:,} DNS queries")
                print(f"  └─ {row['std_above_mean']:.1f} standard deviations above mean")
                print(f"  └─ {row['percentage_above_max']:.1f}% above normal maximum")
                
                # Statistical interpretation
                if row['std_above_mean'] > 10:
                    print(f"  └─ Statistically impossible for normal users (>{row['std_above_mean']:.1f} std)")
    
    return detection_df.drop(columns=['severity_rank'])

# Execute the DNS C&C detection
print("\n" + "="*60)
print("RULE 4: DNS COMMAND & CONTROL (C&C) ACTIVITIES DETECTION")
print("="*60)

baseline_stats_cc, thresholds_cc = calculate_dns_cc_baseline(normal_data)

if thresholds_cc is not None:
    dns_cc_detections = detect_dns_cc(anomalous_data, thresholds_cc)
    
    if not dns_cc_detections.empty:
        print(f"\n=== SUMMARY ===")
        cc_ips = dns_cc_detections['src_ip'].unique()
        print(f"Unique IPs with DNS C&C patterns: {len(cc_ips)}")
        print(f"IPs: {cc_ips.tolist()}")
        
        # Show critical cases
        critical_cases = dns_cc_detections[dns_cc_detections['severity'] == 'CRITICAL']
        if len(critical_cases) > 0:
            print(f"\nCritical DNS C&C IPs: {critical_cases['src_ip'].tolist()}")
            
            # Show most extreme case
            most_extreme = critical_cases.loc[critical_cases['dns_queries'].idxmax()]
            print(f"\nMost extreme case: {most_extreme['src_ip']} with {most_extreme['dns_queries']:,} queries")
            print(f"  └─ {most_extreme['std_above_mean']:.1f}× normal maximum")
            print(f"  └─ {most_extreme['std_above_mean']:.1f} standard deviations above mean")
    else:
        print(f"\n=== SUMMARY ===")
        print("No DNS C&C activity patterns detected in the test dataset.")

## 2.5 Rule 5: Anomalous External Destinations Detection

In [ ]:
# %% [markdown]
# ## 2.5 Rule 5: Anomalous External Destinations Detection

# %%
def calculate_external_destinations_baseline(df, geo_reader):
    """
    Calculate baseline for external destinations detection using normal traffic data.
    
    Parameters:
    df (pd.DataFrame): Normal traffic data
    geo_reader: GeoIP2 reader object for country lookup
    
    Returns:
    dict: Baseline statistics for external destinations
    """
    import ipaddress
    
    # Step 1: Filter for external destinations only (non-private IPs)
    def is_external_ip(ip):
        try:
            ip_obj = ipaddress.ip_address(ip)
            return not ip_obj.is_private
        except:
            return False
    
    external_data = df[df['dst_ip'].apply(is_external_ip)].copy()
    
    print(f"Total flows in dataset: {len(df):,}")
    print(f"External destination flows: {len(external_data):,}")
    
    if len(external_data) == 0:
        print("No external traffic found in dataset")
        return None
    
    # Step 2: Add country information
    def get_country(ip):
        try:
            response = geo_reader.country(ip)
            return response.country.iso_code
        except:
            return 'Unknown'
    
    external_data['country'] = external_data['dst_ip'].apply(get_country)
    
    # Step 3: Calculate baseline statistics per country
    country_stats = external_data.groupby('country').agg({
        'dst_ip': 'count',  # Flow count
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    country_stats.columns = ['country', 'flow_count', 'total_upload', 'total_download']
    country_stats['total_traffic'] = country_stats['total_upload'] + country_stats['total_download']
    
    # Step 4: Calculate baseline statistics per IP
    ip_country_stats = external_data.groupby(['src_ip', 'country']).agg({
        'dst_ip': 'count',
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    ip_country_stats.columns = ['src_ip', 'country', 'flow_count', 'total_upload', 'total_download']
    ip_country_stats['total_traffic'] = ip_country_stats['total_upload'] + ip_country_stats['total_download']
    
    baseline_stats = {
        'countries_contacted': sorted(country_stats['country'].unique()),
        'total_countries': len(country_stats),
        'country_flow_stats': country_stats,
        'ip_country_stats': ip_country_stats,
        'total_external_flows': len(external_data)
    }
    
    print(f"\n=== EXTERNAL DESTINATIONS BASELINE STATISTICS ===")
    print(f"Total countries contacted: {baseline_stats['total_countries']}")
    print(f"Countries contacted: {', '.join(baseline_stats['countries_contacted'])}")
    print(f"Total external flows: {baseline_stats['total_external_flows']:,}")
    
    return baseline_stats

def detect_anomalous_external_destinations(test_df, baseline_stats, geo_reader):
    """
    Apply anomalous external destinations detection rules to test data.
    
    Parameters:
    test_df (pd.DataFrame): Test data to analyze
    baseline_stats (dict): Baseline statistics from normal data
    geo_reader: GeoIP2 reader object for country lookup
    
    Returns:
    dict: Detection results with new countries and traffic anomalies
    """
    import ipaddress
    
    if baseline_stats is None:
        print("No baseline statistics provided - cannot perform detection")
        return {}
    
    # Step 1: Filter for external destinations only (non-private IPs)
    def is_external_ip(ip):
        try:
            ip_obj = ipaddress.ip_address(ip)
            return not ip_obj.is_private
        except:
            return False
    
    external_data = test_df[test_df['dst_ip'].apply(is_external_ip)].copy()
    
    if len(external_data) == 0:
        print("No external traffic found in test dataset")
        return {}
    
    # Step 2: Add country information
    def get_country(ip):
        try:
            response = geo_reader.country(ip)
            return response.country.iso_code
        except:
            return 'Unknown'
    
    external_data['country'] = external_data['dst_ip'].apply(get_country)
    
    # Step 3: Calculate test statistics per country
    test_country_stats = external_data.groupby('country').agg({
        'dst_ip': 'count',
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    test_country_stats.columns = ['country', 'flow_count', 'total_upload', 'total_download']
    test_country_stats['total_traffic'] = test_country_stats['total_upload'] + test_country_stats['total_download']
    
    # Step 4: Calculate test statistics per IP
    test_ip_country_stats = external_data.groupby(['src_ip', 'country']).agg({
        'dst_ip': 'count',
        'up_bytes': 'sum',
        'down_bytes': 'sum'
    }).reset_index()
    
    test_ip_country_stats.columns = ['src_ip', 'country', 'flow_count', 'total_upload', 'total_download']
    test_ip_country_stats['total_traffic'] = test_ip_country_stats['total_upload'] + test_ip_country_stats['total_download']
    
    # Step 5: Detect anomalies
    baseline_countries = set(baseline_stats['countries_contacted'])
    test_countries = set(test_country_stats['country'].unique())
    
    # Find new countries (not in baseline)
    new_countries = test_countries - baseline_countries
    
    # Find countries with significant traffic increases
    baseline_country_flows = baseline_stats['country_flow_stats'].set_index('country')['flow_count'].to_dict()
    
    significant_increases = []
    for _, row in test_country_stats.iterrows():
        country = row['country']
        test_flows = row['flow_count']
        
        if country in baseline_country_flows:
            baseline_flows = baseline_country_flows[country]
            if baseline_flows > 0:
                increase_percentage = ((test_flows - baseline_flows) / baseline_flows) * 100
                if increase_percentage >= 100:  # 100% increase threshold
                    significant_increases.append({
                        'country': country,
                        'baseline_flows': baseline_flows,
                        'test_flows': test_flows,
                        'increase_percentage': increase_percentage,
                        'traffic_mb': row['total_traffic'] / (1024 * 1024)
                    })
    
    # Step 6: Identify IPs involved in anomalous destinations
    anomalous_ips = set()
    
    # IPs accessing new countries
    new_country_ips = {}
    for country in new_countries:
        if country != 'Unknown':
            ips = test_ip_country_stats[test_ip_country_stats['country'] == country]['src_ip'].unique()
            new_country_ips[country] = ips
            anomalous_ips.update(ips)
    
    # IPs contributing to significant traffic increases
    increase_ips = {}
    for increase in significant_increases:
        country = increase['country']
        ips = test_ip_country_stats[test_ip_country_stats['country'] == country]['src_ip'].unique()
        increase_ips[country] = ips
        anomalous_ips.update(ips)
    
    # Step 7: Prepare results
    results = {
        'new_countries': new_countries,
        'new_country_ips': new_country_ips,
        'significant_increases': significant_increases,
        'increase_ips': increase_ips,
        'anomalous_ips': list(anomalous_ips),
        'test_countries': test_countries,
        'baseline_countries': baseline_countries
    }
    
    # Step 8: Print detection results
    print(f"\n=== ANOMALOUS EXTERNAL DESTINATIONS DETECTION RESULTS ===")
    print(f"Baseline countries: {len(baseline_countries)}")
    print(f"Test day countries: {len(test_countries)}")
    print(f"New countries detected: {len(new_countries)}")
    print(f"Countries with significant increases: {len(significant_increases)}")
    print(f"Total anomalous IPs: {len(anomalous_ips)}")
    
    # Report new countries with detailed IP analysis
    if new_countries:
        print(f"\n=== NEW COUNTRIES DETECTED ===")
        
        # Focus on geopolitically sensitive countries
        geopolitical_countries = {'RU', 'IR', 'CN', 'KP', 'BY'}
        sensitive_new_countries = geopolitical_countries.intersection(new_countries)
        
        if sensitive_new_countries:
            print(f"\n⚠️  GEOPOLITICALLY SENSITIVE NEW COUNTRIES:")
            
            # Create consolidated table for IPs accessing new countries
            if 'RU' in sensitive_new_countries and 'IR' in sensitive_new_countries:
                print(f"\n=== DETAILED IP ANALYSIS ===")
                print(f"Category 1: IPs Accessing New Countries ({len(set(new_country_ips.get('RU', [])) | set(new_country_ips.get('IR', [])))} IPs)")
                print()
                print("IP Address | Russia Flows | Iran Flows | Combined Traffic (MB)")
                print("-" * 65)
                
                # Get all IPs that accessed either RU or IR
                all_new_country_ips = set(new_country_ips.get('RU', [])) | set(new_country_ips.get('IR', []))
                
                for ip in sorted(all_new_country_ips):
                    # Get Russia flows
                    ru_data = test_ip_country_stats[(test_ip_country_stats['src_ip'] == ip) & 
                                                   (test_ip_country_stats['country'] == 'RU')]
                    ru_flows = ru_data['flow_count'].iloc[0] if not ru_data.empty else 0
                    
                    # Get Iran flows
                    ir_data = test_ip_country_stats[(test_ip_country_stats['src_ip'] == ip) & 
                                                   (test_ip_country_stats['country'] == 'IR')]
                    ir_flows = ir_data['flow_count'].iloc[0] if not ir_data.empty else 0
                    
                    # Calculate combined traffic
                    combined_traffic = 0
                    if not ru_data.empty:
                        combined_traffic += ru_data['total_traffic'].iloc[0]
                    if not ir_data.empty:
                        combined_traffic += ir_data['total_traffic'].iloc[0]
                    combined_traffic_mb = combined_traffic / (1024 * 1024)
                    
                    print(f"{ip} | {ru_flows} | {ir_flows} | {combined_traffic_mb:.1f}")
            
            # Report individual country summaries
            for country in sorted(sensitive_new_countries):
                if country != 'Unknown':
                    country_data = test_country_stats[test_country_stats['country'] == country].iloc[0]
                    flows = country_data['flow_count']
                    traffic_mb = country_data['total_traffic'] / (1024 * 1024)
                    ips = new_country_ips[country]
                    print(f"\n{country}: {len(ips)} IPs, {flows} flows, {traffic_mb:.1f} MB")
        
        # Report other new countries
        other_new_countries = new_countries - sensitive_new_countries
        if other_new_countries:
            print(f"\nOther new countries:")
            for country in sorted(other_new_countries):
                if country != 'Unknown':
                    country_data = test_country_stats[test_country_stats['country'] == country].iloc[0]
                    flows = country_data['flow_count']
                    traffic_mb = country_data['total_traffic'] / (1024 * 1024)
                    ips = new_country_ips[country]
                    print(f"  └─ {country}: {len(ips)} IPs, {flows} flows, {traffic_mb:.1f} MB")
    
    # Report significant increases with detailed IP analysis
    if significant_increases:
        print(f"\n=== SIGNIFICANT TRAFFIC INCREASES ===")
        print("Countries with >100% traffic increases:")
        
        # Create consolidated table for IPs contributing to traffic increases
        all_increase_ips = []
        for increase in significant_increases:
            country = increase['country']
            baseline_flows = increase['baseline_flows']
            test_flows = increase['test_flows']
            increase_pct = increase['increase_percentage']
            traffic_mb = increase['traffic_mb']
            
            print(f"\n{country}: {baseline_flows} → {test_flows} flows (+{increase_pct:.0f}%) | {traffic_mb:.1f} MB")
            
            # Collect IP data for the consolidated table
            country_ips = test_ip_country_stats[test_ip_country_stats['country'] == country]
            country_ips = country_ips.sort_values('flow_count', ascending=False)
            
            for _, ip_row in country_ips.iterrows():
                all_increase_ips.append({
                    'ip': ip_row['src_ip'],
                    'country': country,
                    'flows': ip_row['flow_count']
                })
        
        # Print consolidated table
        if all_increase_ips:
            print(f"\n=== DETAILED IP ANALYSIS ===")
            print(f"Category 2: IPs Contributing to Anomalous Country Traffic Increases")
            print()
            print("IP Address | Country | Flow Count")
            print("-" * 40)
            
            # Sort by flow count descending
            all_increase_ips.sort(key=lambda x: x['flows'], reverse=True)
            
            for ip_data in all_increase_ips:
                print(f"{ip_data['ip']} | {ip_data['country']} | {ip_data['flows']}")
    
    # Report IPs involved in anomalous destinations
    if anomalous_ips:
        print(f"\n=== ANOMALOUS IPS SUMMARY ===")
        print(f"Total IPs with anomalous external destinations: {len(anomalous_ips)}")
        print(f"IPs: {sorted(anomalous_ips)}")
    
    return results

# Execute the anomalous external destinations detection
print("\n" + "="*60)
print("RULE 5: ANOMALOUS EXTERNAL DESTINATIONS DETECTION")
print("="*60)

baseline_stats_destinations = calculate_external_destinations_baseline(normal_data, geoCC)

if baseline_stats_destinations is not None:
    destinations_detections = detect_anomalous_external_destinations(anomalous_data, baseline_stats_destinations, geoCC)
    
    if destinations_detections and destinations_detections.get('anomalous_ips'):
        print(f"\n=== SUMMARY ===")
        print(f"New countries accessed: {len(destinations_detections['new_countries'])}")
        print(f"Countries with significant increases: {len(destinations_detections['significant_increases'])}")
        print(f"Unique IPs with anomalous destinations: {len(destinations_detections['anomalous_ips'])}")
        
        # Show overlap with other rules
        all_anomalous_ips = set(destinations_detections['anomalous_ips'])
        print(f"\nIPs accessing anomalous destinations: {sorted(all_anomalous_ips)}")
        
        # Highlight geopolitical concerns
        geopolitical_countries = {'RU', 'IR', 'CN', 'KP', 'BY'}  # Russia, Iran, China, North Korea, Belarus
        concerning_countries = geopolitical_countries.intersection(destinations_detections['new_countries'])
        if concerning_countries:
            print(f"\n⚠️  GEOPOLITICAL CONCERNS: New access to {sorted(concerning_countries)}")
    else:
        print(f"\n=== SUMMARY ===")
        print("No anomalous external destinations detected in the test dataset.")

## 2.6 Rule 6: Suspicious External Access Patterns Detection

In [ ]:
def analyze_server_access_baseline(servers_df):
    """
    Analyze external access patterns to corporate servers to establish baseline behavior.
    
    Parameters:
    servers_df (pd.DataFrame): Server access data with external clients
    
    Returns:
    dict: Baseline statistics for external access patterns
    """
    
    print(f"Total flows in servers dataset: {len(servers_df):,}")
    
    # Step 1: Identify corporate servers
    corporate_servers = servers_df['dst_ip'].unique()
    print(f"Corporate servers identified: {sorted(corporate_servers)}")
    
    # Step 2: Calculate time intervals between flows for each external IP
    external_ips = servers_df['src_ip'].unique()
    print(f"External clients accessing servers: {len(external_ips)}")
    
    # Step 3: Calculate access patterns for each external IP
    ip_patterns = []
    
    for ip in external_ips:
        ip_flows = servers_df[servers_df['src_ip'] == ip].copy()
        ip_flows = ip_flows.sort_values('timestamp')
        
        if len(ip_flows) < 10:  # Skip IPs with too few flows for meaningful analysis
            continue
        
        # Calculate time intervals between consecutive flows (in seconds)
        timestamps = ip_flows['timestamp'].values
        intervals = np.diff(timestamps) / 100.0  # Convert to seconds
        
        if len(intervals) > 0 and np.mean(intervals) > 0:
            mean_interval = np.mean(intervals)
            std_interval = np.std(intervals)
            cv = (std_interval / mean_interval) * 100  # Coefficient of Variation
            
            ip_patterns.append({
                'src_ip': ip,
                'flow_count': len(ip_flows),
                'mean_interval': mean_interval,
                'std_interval': std_interval,
                'cv': cv,
                'total_upload': ip_flows['up_bytes'].sum(),
                'total_download': ip_flows['down_bytes'].sum(),
                'target_servers': ip_flows['dst_ip'].unique().tolist()
            })
    
    # Step 4: Calculate baseline statistics
    pattern_df = pd.DataFrame(ip_patterns)
    
    if len(pattern_df) == 0:
        print("No external IPs with sufficient flows for analysis")
        return None
    
    cv_values = pattern_df['cv']
    interval_values = pattern_df['mean_interval']
    
    baseline_stats = {
        'total_external_ips': len(external_ips),
        'analyzed_ips': len(pattern_df),
        'mean_cv': cv_values.mean(),
        'median_cv': cv_values.median(),
        'std_cv': cv_values.std(),
        'min_cv': cv_values.min(),
        'max_cv': cv_values.max(),
        'q5_cv': cv_values.quantile(0.05),
        'q95_cv': cv_values.quantile(0.95),
        'q25_cv': cv_values.quantile(0.25),
        'q75_cv': cv_values.quantile(0.75),
        'mean_interval': interval_values.mean(),
        'median_interval': interval_values.median(),
        'pattern_data': pattern_df
    }
    
    # Print baseline results
    print("\n=== EXTERNAL ACCESS BASELINE STATISTICS ===")
    print(f"Sample size: {len(pattern_df)} external IPs")
    print(f"Mean CV: {baseline_stats['mean_cv']:.3f}, Median CV: {baseline_stats['median_cv']:.3f}")
    print(f"Standard deviation: {baseline_stats['std_cv']:.3f}")
    print(f"Normal range: {baseline_stats['q5_cv']:.3f} - {baseline_stats['q95_cv']:.3f}")
    print(f"25th percentile: {baseline_stats['q25_cv']:.3f}, 75th percentile: {baseline_stats['q75_cv']:.3f}")
    print(f"Average interval between flows: {baseline_stats['mean_interval']:.1f} seconds")
    
    return baseline_stats

def detect_suspicious_external_access(servers_df, baseline_stats=None):
    """
    Detect suspicious external access patterns including bot behavior and slow access attacks.
    
    Parameters:
    servers_df (pd.DataFrame): Server access data to analyze
    baseline_stats (dict): Optional baseline statistics (if None, will calculate from data)
    
    Returns:
    pd.DataFrame: Detected anomalous external IPs
    """
    
    # If no baseline provided, calculate from the data
    if baseline_stats is None:
        baseline_stats = analyze_server_access_baseline(servers_df)
        if baseline_stats is None:
            return pd.DataFrame()
    
    # Extract pattern data
    pattern_df = baseline_stats['pattern_data'].copy()
    
    # Set detection thresholds based on baseline
    normal_min_cv = baseline_stats['q5_cv']
    normal_max_cv = baseline_stats['q95_cv']
    mean_interval = baseline_stats['mean_interval']
    
    # Conservative thresholds for bot detection
    critical_bot_threshold = normal_min_cv * 0.5  # 50% of normal minimum
    high_bot_threshold = normal_min_cv * 0.8     # 80% of normal minimum
    
    # Threshold for slow access patterns
    slow_access_threshold = mean_interval * 3     # 3× normal average interval
    
    print(f"\n=== DETECTION THRESHOLDS ===")
    print(f"Normal CV Range: [{normal_min_cv:.3f}, {normal_max_cv:.3f}]")
    print(f"Critical Bot Threshold: < {critical_bot_threshold:.3f}")
    print(f"High Bot Threshold: < {high_bot_threshold:.3f}")
    print(f"Slow Access Threshold: > {slow_access_threshold:.1f}s (3× normal average of {mean_interval:.1f}s)")
    
    # Detect anomalies
    detections = []
    
    for _, row in pattern_df.iterrows():
        cv = row['cv']
        mean_int = row['mean_interval']
        
        # Detect bot patterns (extremely consistent timing)
        if cv < critical_bot_threshold:
            severity = 'CRITICAL'
            anomaly_type = 'BOT_PATTERN'
            description = f'Extremely consistent timing (CV={cv:.3f})'
        elif cv < high_bot_threshold:
            severity = 'HIGH'
            anomaly_type = 'BOT_PATTERN'
            description = f'Highly consistent timing (CV={cv:.3f})'
        # Detect slow access patterns
        elif mean_int > slow_access_threshold:
            severity = 'HIGH'
            anomaly_type = 'SLOW_ACCESS'
            description = f'Suspicious slow access ({mean_int:.1f}s intervals)'
        else:
            continue
        
        # Calculate percentage deviation
        if anomaly_type == 'BOT_PATTERN':
            percentage_below = ((normal_min_cv - cv) / normal_min_cv) * 100
            deviation_desc = f'{percentage_below:.1f}% below normal minimum'
        else:  # SLOW_ACCESS
            percentage_above = ((mean_int - mean_interval) / mean_interval) * 100
            deviation_desc = f'{percentage_above:.1f}% above normal average'
        
        detections.append({
            'src_ip': row['src_ip'],
            'cv': cv,
            'mean_interval': mean_int,
            'flow_count': row['flow_count'],
            'severity': severity,
            'anomaly_type': anomaly_type,
            'description': description,
            'deviation': deviation_desc,
            'target_servers': row['target_servers']
        })
    
    # Convert to DataFrame
    if not detections:
        print("\nNo external access anomalies detected")
        return pd.DataFrame()
    
    detection_df = pd.DataFrame(detections)
    
    # Sort by severity and type
    severity_order = {'CRITICAL': 4, 'HIGH': 3}
    type_order = {'BOT_PATTERN': 2, 'SLOW_ACCESS': 1}
    detection_df['severity_rank'] = detection_df['severity'].map(severity_order)
    detection_df['type_rank'] = detection_df['anomaly_type'].map(type_order)
    detection_df = detection_df.sort_values(['severity_rank', 'type_rank', 'cv'], ascending=[False, False, True])
    
    # Print detection results
    print(f"\n=== EXTERNAL ACCESS ANOMALY DETECTION RESULTS ===")
    print(f"Total external IPs analyzed: {len(pattern_df)}")
    print(f"Anomalous external IPs detected: {len(detection_df)}")
    
    # Group by anomaly type
    bot_patterns = detection_df[detection_df['anomaly_type'] == 'BOT_PATTERN']
    slow_patterns = detection_df[detection_df['anomaly_type'] == 'SLOW_ACCESS']
    
    if len(bot_patterns) > 0:
        print(f"\nBot Pattern Detections: {len(bot_patterns)}")
        critical_bots = bot_patterns[bot_patterns['severity'] == 'CRITICAL']
        high_bots = bot_patterns[bot_patterns['severity'] == 'HIGH']
        if len(critical_bots) > 0:
            print(f"  - CRITICAL: {len(critical_bots)}")
        if len(high_bots) > 0:
            print(f"  - HIGH: {len(high_bots)}")
    
    if len(slow_patterns) > 0:
        print(f"\nSlow Access Detections: {len(slow_patterns)}")
    
    # Detailed results
    print(f"\n=== DETAILED DETECTION RESULTS ===")
    
    # Critical bot patterns
    if len(bot_patterns[bot_patterns['severity'] == 'CRITICAL']) > 0:
        print("\nCritical Bot Patterns (Extremely Consistent Timing):")
        print("IP Address | CV | Avg Int. (s) | Flows | Target")
        print("-" * 60)
        for _, row in bot_patterns[bot_patterns['severity'] == 'CRITICAL'].iterrows():
            targets = ', '.join(row['target_servers'])
            print(f"{row['src_ip']} | {row['cv']:.3f} | {row['mean_interval']:.1f} | {row['flow_count']} | {targets}")
    
    # High severity patterns
    high_severity = detection_df[detection_df['severity'] == 'HIGH']
    if len(high_severity) > 0:
        print("\nHigh Severity Patterns:")
        print("IP Address | CV | Avg Int. (s) | Flows | Type | Target")
        print("-" * 70)
        for _, row in high_severity.iterrows():
            targets = ', '.join(row['target_servers'])
            print(f"{row['src_ip']} | {row['cv']:.3f} | {row['mean_interval']:.1f} | {row['flow_count']} | {row['anomaly_type']} | {targets}")
    
    # Analyze coordinated attacks
    if len(detection_df) > 0:
        print(f"\n=== COORDINATED ATTACK ANALYSIS ===")
        
        # Group by target servers
        for server in servers_df['dst_ip'].unique():
            server_attackers = detection_df[detection_df['target_servers'].apply(lambda x: server in x)]
            if len(server_attackers) > 0:
                total_flows = 0
                bot_flows = 0
                slow_flows = 0
                
                print(f"\nServer {server}: {len(server_attackers)} attackers")
                
                for _, attacker in server_attackers.iterrows():
                    total_flows += attacker['flow_count']
                    if attacker['anomaly_type'] == 'BOT_PATTERN':
                        bot_flows += attacker['flow_count']
                    else:
                        slow_flows += attacker['flow_count']
                
                print(f"  - Bot patterns: {len(server_attackers[server_attackers['anomaly_type'] == 'BOT_PATTERN'])} attackers, {bot_flows} flows")
                print(f"  - Slow patterns: {len(server_attackers[server_attackers['anomaly_type'] == 'SLOW_ACCESS'])} attackers, {slow_flows} flows")
                print(f"  - Total: {total_flows} flows")
        
        # Check for coordinated patterns
        print(f"\n=== ATTACK PATTERN ANALYSIS ===")
        
        # Analyze CV distribution for bot patterns
        if len(bot_patterns) > 0:
            cv_mean = bot_patterns['cv'].mean()
            cv_std = bot_patterns['cv'].std()
            print(f"Bot CV statistics: Mean={cv_mean:.3f}, Std={cv_std:.3f}")
            
            if cv_std < 0.01:  # Very similar CV values indicate same tool
                print("⚠️  Extremely similar CV values suggest coordinated bot attack using same tool")
        
        # Check for network patterns
        unique_ips = detection_df['src_ip'].unique()
        if len(unique_ips) > 1:
            # Extract network prefixes
            networks = {}
            for ip in unique_ips:
                network = '.'.join(ip.split('.')[:3])
                if network not in networks:
                    networks[network] = []
                networks[network].append(ip)
            
            # Report networks with multiple attackers
            for network, ips in networks.items():
                if len(ips) > 1:
                    print(f"Network {network}.x: {len(ips)} attacking IPs (likely botnet)")
    
    # Summary
    if len(detection_df) > 0:
        print(f"\n=== SUMMARY ===")
        print(f"Total anomalous external IPs: {len(detection_df)}")
        print(f"Critical bot patterns: {len(detection_df[detection_df['severity'] == 'CRITICAL'])}")
        print(f"High severity patterns: {len(detection_df[detection_df['severity'] == 'HIGH'])}")
        
        # Show all detected IPs
        print(f"\nDetected malicious external IPs: {sorted(detection_df['src_ip'].tolist())}")
        
        # Highlight most severe cases
        most_severe = detection_df[detection_df['severity'] == 'CRITICAL']
        if len(most_severe) > 0:
            print(f"\nCritical threat IPs (immediate action required): {sorted(most_severe['src_ip'].tolist())}")
    
    return detection_df.drop(columns=['severity_rank', 'type_rank'])

# Execute Rule 6: External Access Pattern Detection
print("\n" + "="*60)
print("RULE 6: SUSPICIOUS EXTERNAL ACCESS PATTERNS DETECTION")
print("="*60)

# Analyze the servers dataset
server_baseline = analyze_server_access_baseline(servers_data)

if server_baseline is not None:
    # Detect anomalies in the same dataset (as it may contain attacks)
    external_detections = detect_suspicious_external_access(servers_data, server_baseline)
    
    if not external_detections.empty:
        print(f"\n=== EXTERNAL THREAT ASSESSMENT ===")
        print(f"Detected {len(external_detections)} malicious external IPs")
        
        # Severity breakdown
        critical_count = len(external_detections[external_detections['severity'] == 'CRITICAL'])
        high_count = len(external_detections[external_detections['severity'] == 'HIGH'])
        
        print(f"Severity distribution: {critical_count} CRITICAL, {high_count} HIGH")
        
        # Type breakdown
        bot_count = len(external_detections[external_detections['anomaly_type'] == 'BOT_PATTERN'])
        slow_count = len(external_detections[external_detections['anomaly_type'] == 'SLOW_ACCESS'])
        
        print(f"Attack types: {bot_count} bot patterns, {slow_count} slow access patterns")
    else:
        print(f"\n=== EXTERNAL THREAT ASSESSMENT ===")
        print("No suspicious external access patterns detected in the servers dataset.")
